# 04 · Detector clásico con OpenCV (color HSV + watershed)

Este notebook adapta el detector del script `Open CV/VisionEsteroscopicaV3.py` (función `obtener_deteccion_roja_mejorada`) para usarlo como **método alternativo al modelo YOLO**. En la página web se elige el método con el desplegable **Método de reconocimiento**: *Modelo YOLO11-seg* u *OpenCV*.

**Qué se conserva del script original** (mismos valores):
1. **Máscara de color:** rojo en HSV, en los dos tramos del tono (H 0–8 y 172–180), con S ≥ 150 y V ≥ 100.
2. **Morfología:** cierre grande (35 px) para tapar los reflejos de luz; apertura (5 px) y cierre (11 px) para limpiar.
3. **Transformada de distancia suavizada** (Gauss 25 px) y **máximos locales** separados al menos 45 px: cada máximo es la semilla de un tomate.
4. **Watershed** (`skimage`) para separar tomates que se tocan.
5. **Filtros:** área mínima de 500 px² y regla de **visión total**: se descarta el tomate que toca el borde de la imagen.

**Qué cambia:**
- Los valores en píxeles del script estaban pensados para imágenes de **640 px de ancho**. Para no tener que reajustarlos, la imagen se reduce a 640 px de ancho, se segmenta y las máscaras se vuelven a llevar a la resolución original.
- Cada detección devuelve su **máscara**, no solo el centro, porque el resto del sistema mide la profundidad sobre la máscara.
- La "confianza" de OpenCV es la **redondez**: el área del contorno dividida por el área del círculo mínimo que lo encierra (1 = círculo perfecto). El control *Confianza* de la página descarta las regiones menos redondas que ese valor.
- **La posición 3D ya no usa la fórmula empírica `posicion()`** del script ni el emparejamiento por tolerancia en Y. Las dos cámaras se rectifican con la calibración estéreo y X, Y, Z se calculan igual que con YOLO (notebook 02): disparidad entre los centroides de las máscaras izquierda y derecha, con SGBM como respaldo. Así **los dos métodos solo difieren en el reconocimiento** y se pueden comparar de forma justa.

> El detector OpenCV **no necesita GPU ni entrenamiento**, pero depende del color: solo reconoce tomates **rojos** (maduros) y le afectan la iluminación y los objetos rojos del fondo (una manzana roja también se detecta).

In [1]:
# ---------------------------------------------------------------- configuración
import importlib.util
import math
import os
import subprocess
import sys
from dataclasses import dataclass

if "DEMO" not in globals():
    DEMO = True  # 03_api lo pone en False antes de cargar este notebook
_DEMO_04 = DEMO

try:
    import google.colab  # noqa: F401
    _EN_COLAB = True
except ImportError:
    _EN_COLAB = False

if importlib.util.find_spec("skimage") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "scikit-image"], check=True)
from skimage.segmentation import watershed

# Si se abre este notebook solo, primero se cargan las funciones del 02 (calibración, SGBM, localización).
if "procesar_par" not in globals():
    if _EN_COLAB:
        from google.colab import drive
        if not os.path.isdir("/content/drive/MyDrive"):
            drive.mount("/content/drive")
        NB02 = "/content/drive/MyDrive/tomates3d/notebooks/02_ejecucion.ipynb"
    else:
        NB02 = "02_ejecucion.ipynb"
    DEMO = False
    get_ipython().run_line_magic("run", f'-i "{NB02}"')
DEMO = _DEMO_04

import cv2
import numpy as np
print("Funciones del notebook 02 cargadas:", "procesar_par" in globals())

MessageError: Error: credential propagation was unsuccessful

## 1. Detector OpenCV

`OpenCVTomatoDetector` tiene la misma interfaz que `TomatoDetector` (notebook 02): se llama con una imagen BGR y devuelve una lista de `Detection(conf, bbox, mask)`. Por eso `procesar_par()` lo usa sin cambios.

Los parámetros están en `ParametrosOpenCV`, con los valores del script. Están expresados en píxeles de una imagen de 640 px de ancho.

In [ ]:
@dataclass
class ParametrosOpenCV:
    # Rangos HSV del rojo (en OpenCV el tono H va de 0 a 180, así que el rojo aparece en los dos extremos)
    rojo_bajo1: tuple = (0, 150, 100)
    rojo_alto1: tuple = (8, 255, 255)
    rojo_bajo2: tuple = (172, 150, 100)
    rojo_alto2: tuple = (180, 255, 255)
    kernel_brillo: int = 35             # cierre grande: tapa los huecos de los reflejos de luz
    kernel_apertura: int = 5            # elimina puntos sueltos
    kernel_cierre: int = 11             # suaviza el contorno
    blur_distancia: int = 25            # une los picos de un tomate alargado en uno solo
    distancia_minima_centros: int = 45  # separación mínima entre los centros de dos tomates
    grosor_minimo: float = 15           # la semilla debe estar al menos a 15 px del borde de la máscara
    area_minima: float = 500            # px², descarta manchas chicas
    margen: int = 5                     # visión total: se descarta lo que toca el borde
    ancho_trabajo: int = 640            # resolución para la que se ajustaron los valores de arriba


def _elipse(n: int) -> np.ndarray:
    return cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (n, n))


def _impar(n: int) -> int:
    return n if n % 2 else n + 1


class OpenCVTomatoDetector:
    proxy = False

    def __init__(self, params: ParametrosOpenCV | None = None):
        self.p = params or ParametrosOpenCV()

    def info(self) -> dict:
        return {"method": "opencv", "weights": "OpenCV HSV + watershed", "proxy": False, "gpu": False}

    # --- pasos (a la resolución de trabajo) ---
    def mascara_color(self, img: np.ndarray) -> np.ndarray:
        p = self.p
        hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
        m = cv2.inRange(hsv, np.array(p.rojo_bajo1), np.array(p.rojo_alto1)) | \
            cv2.inRange(hsv, np.array(p.rojo_bajo2), np.array(p.rojo_alto2))
        m = cv2.morphologyEx(m, cv2.MORPH_CLOSE, _elipse(p.kernel_brillo))
        m = cv2.morphologyEx(m, cv2.MORPH_OPEN, _elipse(p.kernel_apertura))
        return cv2.morphologyEx(m, cv2.MORPH_CLOSE, _elipse(p.kernel_cierre))

    def separar(self, mascara: np.ndarray) -> tuple[np.ndarray, int, np.ndarray]:
        # Devuelve (etiquetas del watershed, cantidad de semillas, transformada de distancia suavizada).
        p = self.p
        dist = cv2.distanceTransform(mascara, cv2.DIST_L2, 5)
        k = _impar(p.blur_distancia)
        dist_blur = cv2.GaussianBlur(dist, (k, k), 0)
        dist_dilatada = cv2.dilate(dist_blur, _elipse(p.distancia_minima_centros))
        semillas = ((dist_blur == dist_dilatada) & (dist_blur > p.grosor_minimo)).astype(np.uint8)
        semillas = cv2.dilate(semillas, _elipse(5))
        n, marcadores = cv2.connectedComponents(semillas)
        etiquetas = watershed(-dist, marcadores, mask=mascara > 0)
        return etiquetas, n - 1, dist_blur

    def pasos(self, img: np.ndarray) -> dict:
        # Imágenes intermedias (para mostrar cómo trabaja el detector).
        small, _ = self._reducir(img)
        mascara = self.mascara_color(small)
        etiquetas, n, dist_blur = self.separar(mascara)
        return {"imagen": small, "mascara": mascara, "distancia": dist_blur, "etiquetas": etiquetas, "semillas": n}

    def _reducir(self, img: np.ndarray) -> tuple[np.ndarray, float]:
        h, w = img.shape[:2]
        s = self.p.ancho_trabajo / w
        if abs(s - 1) < 1e-6:
            return img, 1.0
        interp = cv2.INTER_AREA if s < 1 else cv2.INTER_LINEAR
        return cv2.resize(img, (self.p.ancho_trabajo, round(h * s)), interpolation=interp), s

    # --- interfaz de detector ---
    def __call__(self, img: np.ndarray, conf: float = 0.35) -> list[Detection]:
        p = self.p
        h, w = img.shape[:2]
        small, s = self._reducir(img)
        hs, ws = small.shape[:2]
        mascara = self.mascara_color(small)
        etiquetas, n, _ = self.separar(mascara)
        out = []
        for etiqueta in range(1, n + 1):
            region = ((etiquetas == etiqueta) & (mascara > 0)).astype(np.uint8)
            contornos, _ = cv2.findContours(region, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
            if not contornos:
                continue
            contorno = max(contornos, key=cv2.contourArea)
            area = cv2.contourArea(contorno)
            if area <= p.area_minima:
                continue
            x, y, bw, bh = cv2.boundingRect(contorno)
            # Regla de visión total: el tomate no puede tocar el borde de la imagen.
            if not (x > p.margen and y > p.margen and x + bw < ws - p.margen and y + bh < hs - p.margen):
                continue
            _, radio = cv2.minEnclosingCircle(contorno)
            redondez = float(min(1.0, area / (math.pi * radio * radio)))
            if redondez < conf:
                continue
            solo = np.zeros_like(region)
            cv2.drawContours(solo, [contorno], -1, 1, -1)
            mask = cv2.resize(solo, (w, h), interpolation=cv2.INTER_NEAREST) > 0
            bbox = (x / s, y / s, (x + bw) / s, (y + bh) / s)
            out.append(Detection(redondez, tuple(float(v) for v in bbox), mask))
        return out


print("OpenCVTomatoDetector definido.")

## 2. Prueba con una imagen sintética

Tres "tomates" rojos (dos de ellos tocándose), uno verde y uno naranja, sobre un fondo con ruido. Lo esperado:
- el watershed separa los dos rojos que se tocan;
- el verde y el naranja no se detectan, porque quedan fuera del rango de rojo.

In [ ]:
def imagen_prueba_opencv(w=1280, h=720, seed=0) -> tuple[np.ndarray, list]:
    rng = np.random.default_rng(seed)
    img = np.clip(rng.normal(150, 20, (h, w, 3)), 0, 255).astype(np.uint8)
    rojos = [(380, 380, 110), (570, 400, 105), (950, 300, 90)]  # los dos primeros se tocan
    for cx, cy, r in rojos:
        cv2.circle(img, (cx, cy), r, (30, 30, 200), -1)
        cv2.circle(img, (cx - r // 3, cy - r // 3), r // 6, (210, 210, 255), -1)  # reflejo de luz
    cv2.circle(img, (900, 580), 80, (40, 160, 60), -1)   # verde
    cv2.circle(img, (200, 600), 70, (20, 120, 240), -1)  # naranja
    return img, rojos


if DEMO:
    import matplotlib.pyplot as plt
    det_cv = OpenCVTomatoDetector()
    img, rojos = imagen_prueba_opencv()
    dets = det_cv(img)
    print(f"Detectados {len(dets)} de {len(rojos)} tomates rojos:")
    for d in dets:
        ys, xs = np.nonzero(d.mask)
        print(f"  centro ({xs.mean():.0f}, {ys.mean():.0f}) px  redondez {d.conf:.2f}")
    pasos = det_cv.pasos(img)
    fig, ax = plt.subplots(1, 4, figsize=(20, 4))
    ax[0].imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB)); ax[0].set_title("imagen")
    ax[1].imshow(pasos["mascara"], cmap="gray"); ax[1].set_title("máscara de rojo")
    ax[2].imshow(pasos["distancia"], cmap="magma"); ax[2].set_title("distancia suavizada")
    ax[3].imshow(pasos["etiquetas"], cmap="nipy_spectral"); ax[3].set_title(f"watershed: {pasos['semillas']} semillas")
    for a in ax:
        a.axis("off")
    plt.show()

## 3. Prueba 3D sintética: OpenCV + calibración estéreo

Usa la escena sintética del notebook 02 (cámaras con calibración conocida) y pinta el disco del tomate de rojo. El detector OpenCV encuentra el tomate en las dos imágenes y `procesar_par()` calcula X, Y, Z igual que con YOLO.

In [ ]:
def escena_tomate_rojo(calib, Z, X, Y, radio=0.035):
    L, R, mL, mR = escena_tomate(calib, Z, X, Y, radio)
    for img, m in ((L, mL), (R, mR)):
        lum = img[m][:, 0].astype(np.float32)  # la textura del disco, en gris
        img[m] = np.stack([0.15 * lum, 0.12 * lum, np.clip(110 + 0.55 * lum, 0, 255)], axis=1).astype(np.uint8)
    return L, R


def prueba_opencv_3d(calib, distancias=(0.30, 0.45, 0.60)) -> list[dict]:
    det_cv = OpenCVTomatoDetector()
    filas = []
    for Z in distancias:
        X, Y = calib.baseline / 2, 0.02
        L, R = escena_tomate_rojo(calib, Z, X, Y)
        res = procesar_par(L, R, calib, det_cv, imagenes=False)
        if not res["tomatoes"]:
            print(f"Z real {Z:.2f} m -> no detectado")
            filas.append({"Z real (m)": Z, "Z medida (m)": float("nan")})
            continue
        t = res["tomatoes"][0]
        z = (t["xyz_mask_m"] or t["xyz_m"])[2]
        print(f"Z real {Z:.2f} m -> {z:.4f} m ({1000 * (z - Z):+.1f} mm) | método {t['metodo_z']} | "
              f"detección {res['timing_ms']['detect']:.0f} ms")
        filas.append({"Z real (m)": Z, "Z medida (m)": z, "error (mm)": 1000 * (z - Z),
                      "detección (ms)": res["timing_ms"]["detect"]})
    return filas


if DEMO:
    prueba_opencv_3d(calibracion_generica(baseline_m=0.08))

## 4. Comparar YOLO y OpenCV en un par real

Completá las rutas (o subí las imágenes en Colab). Se procesa el mismo par con los dos métodos y se muestran los resultados uno debajo del otro.

In [ ]:
RUTA_IZQ = None  # ej: "capturas/par_L.jpg"
RUTA_DER = None
CONFIANZA = 0.35

if DEMO:
    try:
        calib = cargar_calibracion()
    except FileNotFoundError:
        calib = calibracion_generica(baseline_m=0.08)
        print("No hay calibración guardada; uso:", calib.created)
    if _EN_COLAB and RUTA_IZQ is None:
        from google.colab import files
        print("Elegí la imagen IZQUIERDA"); RUTA_IZQ = next(iter(files.upload()))
        print("Elegí la imagen DERECHA"); RUTA_DER = next(iter(files.upload()))
    if RUTA_IZQ and RUTA_DER:
        izq, der = leer_imagen(RUTA_IZQ), leer_imagen(RUTA_DER)
        for nombre, det in (("YOLO11-seg", TomatoDetector()), ("OpenCV", OpenCVTomatoDetector())):
            res = procesar_par(izq, der, calib, det, conf=CONFIANZA)
            print(f"\n===== {nombre}: {res['count']} tomate(s), detección {res['timing_ms']['detect']} ms")
            mostrar_resultado(res)
    else:
        print("Completá RUTA_IZQ y RUTA_DER para comparar los dos métodos en un par real.")